In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path, index_col=0)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df
# Function to split the name column and create new columns
def split_name_column(name):
    parts = name.split('_')
    parameters = parts[-1].replace('.qasm', '').strip('[]')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'
    


In [ ]:
# Get all the results in a df
folder_path = './results'
df = read_and_merge_csv_files(folder_path)

# Create a column to categorize the input type
df[['Input_type']] = df['Input'].apply(lambda x: pd.Series(x.split('_')[0]))

# Apply the function to the name column and create new columns
df[['Algorithm', 'Qubits_number',  'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

# Create a column to categorize the gate type
df[['Gate_type']] = df['Gate'].apply(lambda x: pd.Series(get_gate_type(x)))

# Calculate position percentage and categorize it
df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
df['position_percentage'] = (df['Position'] / df['max_position']) * 100
df['Relative_position'] = df['position_percentage'].apply(categorize_position)

# Drop the intermediate columns if needed
df = df.drop(columns=['max_position', 'position_percentage'])
df = df.drop(columns=['Name'])

df

In [ ]:
# List of columns related to "Killed" metrics
killed_columns = [col for col in df.columns if col.startswith('Killed_')]

# Calculate the percentage of True values for each "Killed" column
true_percentages = (df[killed_columns].mean() * 100).sort_values(ascending=False)

# Print the results
print("Percentage of True values for each 'Killed' column:")
print(true_percentages)

In [ ]:
def confusionMatrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale='Dense',
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=14)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [ ]:

def printConfusionMatrixes(df_confusion, name):
    confusion_matrix_chisquare = confusionMatrix(df_confusion, 'Killed_IC','Killed_NC')
    confusion_matrix_hellinger = confusionMatrix(df_confusion, 'Killed_IH','Killed_NH')
    confusion_matrix_trace = confusionMatrix(df_confusion, 'Killed_IT','Killed_NT')
    confusion_matrix_fidelity = confusionMatrix(df_confusion, 'Killed_IF','Killed_NF')
    confusion_matrix_jensenshannon = confusionMatrix(df_confusion, 'Killed_IJ','Killed_NJ')
    
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=('Chisquare', 'Hellinger', 'Trace', 'Fidelity', 'Jensen-shannon'), x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05)
    
    # Add heatmaps to subplots
    create_heatmap(fig, confusion_matrix_chisquare, row=1, col=1, showscale=True)
    create_heatmap(fig, confusion_matrix_hellinger, row=1, col=2, showscale=False)
    create_heatmap(fig, confusion_matrix_trace, row=1, col=3, showscale=False)
    create_heatmap(fig, confusion_matrix_fidelity, row=1, col=4, showscale=False)
    create_heatmap(fig, confusion_matrix_fidelity, row=1, col=5, showscale=False)
    
    fig.update_layout(
        title_text=name,
        height=400,
        width=2000,
        showlegend=False
    )
    
    fig.show()


In [ ]:
# OVERALL CONFUSION MATRIX
printConfusionMatrixes(df, 'Overall confusion matrix')

In [ ]:
# Confusion matrixes grouped by qubit numbers
qubit_numbers = df['Qubits_number'].unique()
for x in qubit_numbers:
    df_qubits = df[df['Qubits_number'] == x]
    printConfusionMatrixes(df_qubits, f'Confusion matrix for {x} qubits circuits')

In [ ]:
# Confusion matrixes grouped by algorithms
algorithms = df['Algorithm'].unique()
for x in algorithms:
    df_algorithm = df[df['Algorithm'] == x]
    printConfusionMatrixes(df_algorithm, f'Confusion matrix for {x} algorithm')

In [ ]:
# Confusion matrixes grouped by algorithms
operators = df['Operator'].unique()
for x in operators:
    df_operator = df[df['Operator'] == x]
    printConfusionMatrixes(df_operator, f'Confusion matrix for {x} operator')

In [ ]:
# Confusion matrixes grouped by algorithms
input_types = df['Input_type'].unique()
for x in input_types:
    df_input = df[df['Input_type'] == x]
    printConfusionMatrixes(df_input, f'Confusion matrix for {x} inputs')

In [ ]:
# Confusion matrixes grouped by algorithms
gate_types = df['Gate_type'].unique()
for x in gate_types:
    df_gate = df[df['Gate_type'] == x]
    printConfusionMatrixes(df_gate, f'Confusion matrix for {x} gates')

In [ ]:
# Confusion matrixes grouped by position
relative_positions = df['Relative_position'].unique()
for x in relative_positions:
    df_position = df[df['Relative_position'] == x]
    printConfusionMatrixes(df_position, f'Confusion matrix for mutations in the {x} of the circuit')